# Transformer Algorithm Review & Practice

이 노트북은 흩어져 있던 Transformer 관련 실습을 한 번에 복습하고, 필요할 때 바로 가져다 쓸 수 있도록 다시 정리한 버전입니다.

## 이 노트북의 목표
- Transformer 계열 모델 구조를 빠르게 다시 떠올린다.
- `tokenizer / processor / model / pipeline` 관계를 실습 코드와 연결한다.
- Decoder-only, Encoder-Decoder, chat template, tokenizer, dtype 메모리 개념을 한 흐름으로 정리한다.
- 나중에 새 모델을 테스트할 때 그대로 복사해서 쓸 수 있는 템플릿을 남긴다.

## 원본 자료 출처
- `module03TransformerAlgorithm/01lectureNote.md`
- `module03TransformerAlgorithm/A.핵심개념_Transformer모델기초.md`
- `HF_Exaone_test.ipynb`
- `HF_FLAN_T5_test.ipynb`
- `HF_LLaMA3_Endpoint_Tokenizer_Test.ipynb`
- `LLM_test_SKTGPT2_base_v2.ipynb`
- `Memory_requirements_by_datatype.ipynb`

필요한 부분만 남기고, 중복되거나 나중에 다시 보기 어려운 코드는 정리해서 새로 썼습니다.


## 1. Transformer 핵심 개념 한 장 정리

| 분류 | 학습/생성 방식 | 대표 목적 | 실습 연결 |
| --- | --- | --- | --- |
| Encoder Only | 입력을 양방향으로 이해 | 분류, 이해, 마스킹 복원 | 개념 비교용 |
| Decoder Only | 앞 토큰만 보고 다음 토큰 생성 | 생성, 자동완성, 대화 | KoGPT2, EXAONE |
| Encoder-Decoder | 입력을 읽고 새로운 출력 생성 | 번역, 요약, QA | FLAN-T5 |

### 같이 기억할 개념
- `BF16 / FP16`: 모델 메모리를 줄이고 속도를 높이기 위한 dtype 선택
- `pipeline = processor + model`: 전처리와 추론을 합친 사용 인터페이스
- `wte`: word/text embedding
- `wpe`: position embedding
- `dropout`: 학습 중 일부 연결을 끄는 규제 기법
- `messages = [{role, content}, ...]`: 멀티턴 채팅 입력의 기본 포맷

### 직관 메모
- Encoder Only: 문장을 이해하는 쪽에 강함
- Decoder Only: 문장을 이어 쓰고 생성하는 쪽에 강함
- Encoder-Decoder: 입력을 다른 형태의 출력으로 바꾸는 데 강함


In [1]:
# 공통 환경 설정
# 무거운 모델은 GPU/토큰 권한이 필요할 수 있으므로, 필요한 셀만 골라 실행하세요.

import warnings
from getpass import getpass

import torch

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device = {device}')
print(f'torch version = {torch.__version__}')


device = cpu
torch version = 2.10.0+cpu


In [2]:
# 공통 유틸 함수
# 여러 실습 노트북에서 반복되던 로직을 복습용으로 단순화했습니다.

from typing import Iterable


def decode_tokens(tokenizer, token_ids, skip_special_tokens=True):
    """
    model.generate() 결과를 사람이 읽기 쉬운 문자열로 바꿉니다.
    tensor / list 입력을 모두 처리합니다.
    """
    if isinstance(token_ids, torch.Tensor):
        if token_ids.ndim == 2:
            token_ids = token_ids[0]
        token_ids = token_ids.detach().cpu().tolist()
    return tokenizer.decode(token_ids, skip_special_tokens=skip_special_tokens)


def bytes_per_param(dtype_name):
    mapping = {
        'fp32': 4,
        'float32': 4,
        'fp16': 2,
        'float16': 2,
        'bf16': 2,
        'bfloat16': 2,
        'int8': 1,
        '4bit': 0.5,
    }
    return mapping[dtype_name.lower()]


def estimate_model_memory(num_params_billion, dtype_name='fp16'):
    """대략적인 파라미터 메모리 사용량을 GB 단위로 계산합니다."""
    total_bytes = num_params_billion * 1_000_000_000 * bytes_per_param(dtype_name)
    return total_bytes / (1024 ** 3)


def show_memory_table(param_sizes=(0.1, 1, 3, 8), dtypes=('fp32', 'fp16', 'bf16', 'int8')):
    for n in param_sizes:
        row = {dtype: round(estimate_model_memory(n, dtype), 2) for dtype in dtypes}
        print(f'{n}B params -> {row}')


show_memory_table()


0.1B params -> {'fp32': 0.37, 'fp16': 0.19, 'bf16': 0.19, 'int8': 0.09}
1B params -> {'fp32': 3.73, 'fp16': 1.86, 'bf16': 1.86, 'int8': 0.93}
3B params -> {'fp32': 11.18, 'fp16': 5.59, 'bf16': 5.59, 'int8': 2.79}
8B params -> {'fp32': 29.8, 'fp16': 14.9, 'bf16': 14.9, 'int8': 7.45}


## 2. dtype와 메모리 감각 익히기

원본 `Memory_requirements_by_datatype.ipynb`의 핵심은 아주 단순합니다.

- 같은 모델이라도 dtype이 바뀌면 필요한 메모리가 달라진다.
- FP32 -> FP16/BF16으로 바꾸면 파라미터 메모리는 대체로 절반 수준이 된다.
- 그래서 실무에서는 `torch_dtype=torch.float16` 또는 `torch.bfloat16`을 자주 쓴다.

아래 셀은 작은 더미 모델로 이 감각을 확인하는 예제입니다.


In [3]:
import torch.nn as nn


class DummyModel(nn.Module):
    """dtype 변화에 따른 메모리 차이를 보기 위한 아주 작은 예제 모델"""

    def __init__(self):
        super().__init__()
        torch.manual_seed(123)
        self.token_embedding = nn.Embedding(8, 16)
        self.linear_1 = nn.Linear(16, 16)
        self.layernorm_1 = nn.LayerNorm(16)
        self.linear_2 = nn.Linear(16, 16)
        self.layernorm_2 = nn.LayerNorm(16)
        self.head = nn.Linear(16, 8)

    def forward(self, x):
        x = self.token_embedding(x)
        x = self.layernorm_1(self.linear_1(x))
        x = self.layernorm_2(self.linear_2(x))
        return self.head(x)


def model_footprint_bytes(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    return param_size + buffer_size


def print_param_dtype(model):
    for name, param in model.named_parameters():
        print(f'{name}: {param.dtype}')


model_fp32 = DummyModel()
model_fp16 = DummyModel().half()

print('fp32 bytes =', model_footprint_bytes(model_fp32))
print('fp16 bytes =', model_footprint_bytes(model_fp16))
print_param_dtype(model_fp16)


fp32 bytes = 3488
fp16 bytes = 1744
token_embedding.weight: torch.float16
linear_1.weight: torch.float16
linear_1.bias: torch.float16
layernorm_1.weight: torch.float16
layernorm_1.bias: torch.float16
linear_2.weight: torch.float16
linear_2.bias: torch.float16
layernorm_2.weight: torch.float16
layernorm_2.bias: torch.float16
head.weight: torch.float16
head.bias: torch.float16


## 3. Hugging Face 사용 구조: tokenizer / processor / model / pipeline

실습 파일을 다시 보면 거의 항상 아래 패턴으로 정리할 수 있습니다.

1. `tokenizer` 또는 `processor`를 로드한다.
2. `model`을 로드한다.
3. 입력을 토큰화한다.
4. `model.generate()` 또는 `pipeline()`으로 결과를 만든다.
5. 디코딩해서 사람이 읽는 텍스트로 바꾼다.

텍스트 전용 모델은 주로 `tokenizer`를 쓰고, 이미지/오디오까지 들어가면 `processor`가 더 자연스럽습니다.


In [4]:
# 가장 기본적인 로딩 템플릿
# 처음 새 모델을 테스트할 때는 이 셀을 복사해서 model_name만 바꿔 쓰면 됩니다.

from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer


def load_tokenizer_and_model(model_name, model_type='causal', torch_dtype='auto'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if model_type == 'causal':
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            device_map='auto',
            trust_remote_code=True,
        )
    elif model_type == 'seq2seq':
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            device_map='auto',
        )
    else:
        raise ValueError("model_type must be 'causal' or 'seq2seq'")

    return tokenizer, model


# 예시
# tokenizer, model = load_tokenizer_and_model('google/flan-t5-base', model_type='seq2seq', torch_dtype=torch.float16)
# tokenizer, model = load_tokenizer_and_model('skt/kogpt2-base-v2', model_type='causal')


## 4. Decoder-Only 실습 1: KoGPT2로 다음 토큰 생성 이해하기

원본 `LLM_test_SKTGPT2_base_v2.ipynb`는 아주 좋은 입문 실습입니다.

여기서 복습 포인트는 두 가지입니다.
- GPT 계열은 입력 뒤에 이어서 텍스트를 생성한다.
- `greedy search`, `beam search`, `sampling` 설정에 따라 결과가 달라진다.


In [5]:
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast


def run_kogpt2_demo(prompt, max_new_tokens=50, num_beams=None):
    model_name = 'skt/kogpt2-base-v2'

    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        model_name,
        eos_token='</s>',
    )
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    model.eval()

    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    generate_kwargs = {
        'do_sample': False,
        'max_new_tokens': max_new_tokens,
    }
    if num_beams:
        generate_kwargs['num_beams'] = num_beams
        generate_kwargs['no_repeat_ngram_size'] = 3

    with torch.no_grad():
        output_ids = model.generate(input_ids, **generate_kwargs)

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# prompt = '조선시대 최고의 장군은 누구야?'
# print(run_kogpt2_demo(prompt, num_beams=None))      # Greedy
# print(run_kogpt2_demo(prompt, num_beams=3))         # Beam Search


## 5. Decoder-Only 실습 2: EXAONE 같은 Instruct 모델을 chat template로 다루기

원본 `HF_Exaone_test.ipynb`의 핵심은 `apply_chat_template()` 입니다.

Instruct 모델은 단순 문자열을 넣기보다, 아래처럼 역할이 있는 메시지 구조로 넣는 것이 훨씬 안정적입니다.

- `system`: 모델의 기본 지침
- `user`: 실제 질문
- `assistant`: 이전 응답(멀티턴일 때)


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_exaone_chat_model(model_name='beomi/EXAONE-3.5-2.4B-Instruct-Llamafied'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype='auto',
        device_map='auto',
        trust_remote_code=True,
    )
    return tokenizer, model


def generate_chat_response(tokenizer, model, user_text, system_text='You are a helpful assistant.', max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': system_text},
        {'role': 'user', 'content': user_text},
    ]

    # 핵심: instruct 계열은 chat template를 먼저 적용한 뒤 토큰화합니다.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            eos_token_id=tokenizer.eos_token_id,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    decoded = decode_tokens(tokenizer, output_ids)
    return decoded


# tokenizer, model = load_exaone_chat_model()
# print(generate_chat_response(tokenizer, model, 'MCP를 5문장으로 설명해줘.'))


## 6. Encoder-Decoder 실습: FLAN-T5로 입력을 변환하기

원본 `HF_FLAN_T5_test.ipynb`는 Encoder-Decoder의 감각을 익히기에 좋습니다.

이 계열은 보통 다음처럼 생각하면 쉽습니다.
- 입력 문장을 이해한다.
- 그 입력을 바탕으로 답, 요약, 번역 같은 새 문장을 생성한다.

즉, Decoder-only가 `이어 쓰기`라면, FLAN-T5는 `입력을 바탕으로 새 출력 만들기`에 더 가깝습니다.


In [7]:
from transformers import T5ForConditionalGeneration, T5Tokenizer


def load_flan_t5(model_name='google/flan-t5-base'):
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = T5ForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map='auto',
    )
    return tokenizer, model


def generate_with_flan_t5(tokenizer, model, prompt, max_new_tokens=128, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# tokenizer, model = load_flan_t5()
# prompts = [
#     'What is the capital of France?',
#     'Think step by step. What is the capital of France?',
#     '대한민국의 수도는 어디야?',
# ]
# for prompt in prompts:
#     print(prompt)
#     print(generate_with_flan_t5(tokenizer, model, prompt))
#     print('-' * 80)


## 7. Tokenizer 감각: encode / decode / padding / truncation

원본 `HF_LLaMA3_Endpoint_Tokenizer_Test.ipynb`의 중요한 포인트는 모델 추론보다 먼저 토크나이저 동작을 몸으로 익히는 것입니다.

아래 셀은 어떤 tokenizer든 공통적으로 확인해볼 수 있는 최소 실습입니다.


In [8]:
from transformers import AutoTokenizer


def inspect_tokenizer(model_name, texts, max_length=12):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print(f'model_name = {model_name}')
    print('-' * 80)

    for text in texts:
        encoded = tokenizer.encode(text)
        decoded = tokenizer.decode(encoded)
        print(f'text     : {text}')
        print(f'encoded  : {encoded}')
        print(f'decoded  : {decoded}')
        print('-' * 80)

    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )

    print('input_ids shape     =', tuple(batch['input_ids'].shape))
    print('attention_mask shape=', tuple(batch['attention_mask'].shape))
    return tokenizer, batch


# texts = ['Hi, how are you?', 'I am good.', '안녕, 잘 지내지?']
# tokenizer, batch = inspect_tokenizer('google/flan-t5-base', texts)


## 8. Multi-turn Chat 포맷 복습

멀티턴 채팅에서 중요한 것은 모델이 아니라 입력 구조입니다.
이전 대화가 누적되어야 맥락이 유지됩니다.

아래 셀은 나중에 어떤 chat 모델을 쓰더라도 재활용할 수 있는 메시지 템플릿입니다.


In [9]:
messages = [
    {'role': 'system', 'content': '당신은 Transformer 학습을 돕는 튜터입니다.'},
    {'role': 'user', 'content': 'Transformer 모델 종류를 표로 정리해줘.'},
    {'role': 'assistant', 'content': 'Encoder Only / Decoder Only / Encoder-Decoder로 나눌 수 있습니다.'},
    {'role': 'user', 'content': '그중 Decoder Only를 예시와 함께 다시 설명해줘.'},
]

for msg in messages:
    print(f"[{msg['role']}] {msg['content']}")

# chat template가 있는 tokenizer라면 보통 이렇게 연결합니다.
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# print(prompt)


[system] 당신은 Transformer 학습을 돕는 튜터입니다.
[user] Transformer 모델 종류를 표로 정리해줘.
[assistant] Encoder Only / Decoder Only / Encoder-Decoder로 나눌 수 있습니다.
[user] 그중 Decoder Only를 예시와 함께 다시 설명해줘.


## 9. 바로 가져다 쓰는 실전 체크리스트

### 새 모델을 테스트할 때 순서
1. 이 모델이 `causal`인지 `seq2seq`인지 먼저 구분한다.
2. GPU 메모리가 충분하지 않으면 `fp16`, `bf16`, `8bit` 여부를 먼저 생각한다.
3. Instruct 모델이면 단순 문자열보다 `chat template` 적용을 우선 검토한다.
4. 토크나이저 encode/decode를 먼저 확인해서 입력 형태를 감각적으로 이해한다.
5. 생성 결과가 이상하면 `max_new_tokens`, `temperature`, `num_beams`, `do_sample`을 조정한다.

### 이 노트북에서 특히 다시 보면 좋은 셀
- dtype 감각: 메모리 관련 셀
- Decoder-only: KoGPT2 / EXAONE 셀
- Encoder-Decoder: FLAN-T5 셀
- Chat format: messages / apply_chat_template 셀
- Tokenizer: inspect_tokenizer 셀

### 마지막 한 줄 요약
Transformer를 잘 다루는 핵심은 모델 이름을 많이 아는 것이 아니라,
`입력 형식`, `모델 구조`, `dtype`, `생성 옵션`을 연결해서 보는 습관입니다.
